In [3]:
import pandas as pd
import numpy as np
from openbabel import pybel
from IPython.display import SVG
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem, Crippen
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from chembl_webresource_client.new_client import new_client
IPythonConsole.ipython_useSVG = True
import matplotlib.pyplot as plt
import seaborn as sns
from padelpy import padeldescriptor
import zipfile
import urllib.request
import os

pd.set_option('display.max_columns', 50)
sns.set_style("whitegrid")
RANDOM_SEED = 42

In [5]:
df = pd.read_csv('data/acetylcholinesterase_activity_05_pic50_filtered.csv')
df.head()

,molecule_chembl_id,canonical_smiles,class,MW,LogP,NumHDonors,NumHAcceptors,TPSA,RotatableBonds,pIC50
0,CHEMBL133897,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,active,312.325,2.8032,0.0,5.0,66.49,6.0,6.124939
1,CHEMBL336398,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,active,376.913,4.5546,0.0,4.0,51.02,4.0,7.000000
2,CHEMBL131588,CN(C(=O)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F)c1ccccc1,inactive,426.851,5.3574,0.0,4.0,51.02,4.0,4.301030
3,CHEMBL130628,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC(F)(F)F,active,404.845,4.7069,0.0,4.0,51.02,3.0,6.522879
4,CHEMBL130478,CSc1nc(-c2ccc(OC(F)(F)F)cc2)nn1C(=O)N(C)C,active,346.334,3.0953,0.0,5.0,60.25,3.0,6.096910


In [10]:
df2 = df[['canonical_smiles', 'molecule_chembl_id']]
df2.head(2)

,canonical_smiles,molecule_chembl_id
0,CCOc1nn(-c2cccc(OCc3ccccc3)c2)c(=O)o1,CHEMBL133897
1,O=C(N1CCCCC1)n1nc(-c2ccc(Cl)cc2)nc1SCC1CC1,CHEMBL336398


In [ ]:
# write df csv to molecule.smi file
temp_smi_file_path = 'data/molecule.smi'
df2.to_csv(temp_smi_file_path, sep='\t', index=False, header=False)

# --- 1. Download PaDEL Fingerprint XML Config Files ---
# PaDEL needs specific XML files to compute explicit fingerprint types
xml_zip_url = "https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip"
zip_path = "data/fingerprints_xml.zip"

if not os.path.exists(zip_path):
    print("Downloading fingerprint XML files...")
    urllib.request.urlretrieve(xml_zip_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/fingerprints_xml")

# Available fingerprinters in this set include:
# 'PubchemFingerprinter.xml', 'MACCSFingerprinter.xml', 'SubstructureFingerprinter.xml',
# 'Fingerprinter.xml' (Standard CDK 1024-bit), 'ExtendedFingerprinter.xml'
fingerprint_xml = os.path.join("data/fingerprints_xml", "PubchemFingerprinter.xml")
padeldescriptor(mol_dir='data/molecule.smi', d_file='data/descriptors.csv')



In [13]:
# 2. Calculate PubChem Fingerprints using padelpy
padeldescriptor(
    mol_dir='data/molecule.smi',
    d_file='data/descriptors_output.csv',
    fingerprints=True,
    descriptortypes='data/fingerprints_xml/PubchemFingerprinter.xml', # Path to your XML config
    removesalt=True,
    standardizenitro=True
)

In [ ]:
# Clean up temporary smiles file if needed
if os.path.exists(temp_smi_file_path):
    os.remove(temp_smi_file_path)

## Prepare X and Y Matrices

In [14]:
df2_x = pd.read_csv('data/descriptors_output.csv')
df2_x.head(2)

,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,PubchemFP10,PubchemFP11,PubchemFP12,PubchemFP13,PubchemFP14,PubchemFP15,PubchemFP16,PubchemFP17,PubchemFP18,PubchemFP19,PubchemFP20,PubchemFP21,PubchemFP22,PubchemFP23,...,PubchemFP856,PubchemFP857,PubchemFP858,PubchemFP859,PubchemFP860,PubchemFP861,PubchemFP862,PubchemFP863,PubchemFP864,PubchemFP865,PubchemFP866,PubchemFP867,PubchemFP868,PubchemFP869,PubchemFP870,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL133897,1,1,1,0,0,0,0,0,0,1,1,1,1,0,1,1,0,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,CHEMBL336398,1,1,1,0,0,0,0,0,0,1,1,1,1,0,1,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [16]:
df2_x.shape
df2_x.columns

Index(['Name', 'PubchemFP0', 'PubchemFP1', 'PubchemFP2', 'PubchemFP3',
       'PubchemFP4', 'PubchemFP5', 'PubchemFP6', 'PubchemFP7', 'PubchemFP8',
       ...
       'PubchemFP871', 'PubchemFP872', 'PubchemFP873', 'PubchemFP874',
       'PubchemFP875', 'PubchemFP876', 'PubchemFP877', 'PubchemFP878',
       'PubchemFP879', 'PubchemFP880'],
      dtype='str', length=882)

In [17]:
df2_x = df2_x.drop(columns=['Name'])
df2_y = df['pIC50']
df3 = pd.concat([df2_x, df2_y], axis=1)

In [18]:
df3.columns

Index(['PubchemFP0', 'PubchemFP1', 'PubchemFP2', 'PubchemFP3', 'PubchemFP4',
       'PubchemFP5', 'PubchemFP6', 'PubchemFP7', 'PubchemFP8', 'PubchemFP9',
       ...
       'PubchemFP872', 'PubchemFP873', 'PubchemFP874', 'PubchemFP875',
       'PubchemFP876', 'PubchemFP877', 'PubchemFP878', 'PubchemFP879',
       'PubchemFP880', 'pIC50'],
      dtype='str', length=882)

In [19]:
df3.to_csv('data/acetylcholinesterase_activity_06_pubchem_pic50_modeldata.csv', index=False)